In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
CLEAN = ROOT / "data" / "processed" / "loans_clean.parquet"

loans = pd.read_parquet(CLEAN)
print(loans.shape)

(2260668, 42)


In [2]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv(ROOT / ".env")
engine = create_engine(os.environ["DATABASE_URL"])

with engine.connect() as conn:
    print(conn.execute(text("SELECT current_database()")).scalar())

lending_db


pandas `to_sql` sends rows as INSERT statements, which is very slow for 2.26 million rows.
Instead, create the empty table with `to_sql`, then stream the rows in with PostgreSQL's `COPY`,
which is built for bulk loads.

In [3]:
import io
import time


def copy_to_postgres(df, table, engine, chunk_size=200_000):
    df.head(0).to_sql(table, engine, if_exists="replace", index=False)
    raw = engine.raw_connection()
    try:
        with raw.cursor() as cur:
            for start in range(0, len(df), chunk_size):
                buffer = io.StringIO()
                df.iloc[start:start + chunk_size].to_csv(buffer, index=False, header=False)
                buffer.seek(0)
                cur.copy_expert(f"COPY {table} FROM STDIN WITH (FORMAT csv)", buffer)
        raw.commit()
    finally:
        raw.close()


start = time.time()
copy_to_postgres(loans, "loans", engine)
print(f"Loaded in {time.time() - start:.0f} seconds")

Loaded in 107 seconds


In [4]:
with engine.begin() as conn:
    conn.execute(text("ALTER TABLE loans ADD PRIMARY KEY (id)"))
    conn.execute(text("CREATE INDEX ON loans (issue_year)"))
    conn.execute(text("CREATE INDEX ON loans (grade)"))
    conn.execute(text("ANALYZE loans"))

schema = pd.read_sql("""
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_name = 'loans'
ORDER BY ordinal_position
""", engine)
print(len(schema), "columns")
schema.groupby("data_type")["column_name"].apply(", ".join).to_frame()

42 columns


,column_name
data_type,
bigint,"id, term, bad"
boolean,"matured, old_policy"
double precision,"loan_amnt, funded_amnt, int_rate, installment,..."
integer,"months_on_book, issue_year"
text,"grade, sub_grade, emp_length, home_ownership, ..."
timestamp without time zone,"issue_d, earliest_cr_line, last_pymnt_d"


Check that PostgreSQL holds exactly what pandas had: same row count, same number of judged loans,
same default rate.

In [5]:
check = pd.read_sql("""
SELECT
    COUNT(*) AS loans,
    COUNT(bad) AS judged,
    ROUND(AVG(bad) * 100, 2) AS default_rate_pct
FROM loans
""", engine)
print(check)

judged = loans[loans["bad"].notna()]
assert check["loans"].iloc[0] == len(loans)
assert check["judged"].iloc[0] == len(judged)
print("pandas default rate:", round(judged["bad"].mean() * 100, 2))

     loans  judged  default_rate_pct
0  2260668  779026             15.19


pandas default rate: 15.19


PostgreSQL now holds all 2,260,668 loans and 42 columns, with a primary key on `id` and indexes on the
columns most queries filter by. Row counts, judged loans, and the 15.19% default rate match pandas
exactly, so the SQL analysis in notebook 04 starts from the same data.